In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import time
import os
import glob
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet50, ResNet50_Weights
import pandas as pd
from datetime import datetime
import threading
from collections import defaultdict
from functools import wraps
from contextlib import contextmanager

# ==========================================
# 1. load_cifar10
# ==========================================
# 固定随机种子，消除实验随机波动，保证结果稳定
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

def load_cifar10(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    # 标准CIFAR10测试集
    full_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
    total_all = len(full_dataset)

    # 约束：使用数据量不超过原始数据集10%
    sample_count = int(total_all * 0.1)
    indices = np.random.permutation(total_all)[:sample_count]
    sub_dataset = Subset(full_dataset, indices)
    print(f"完整数据集总量：{total_all}，本次测试选用样本：{sample_count}，数据占比：{sample_count / total_all:.2%}")
    loader = DataLoader(sub_dataset, batch_size=batch_size, shuffle=False, drop_last=False)
    return loader

# ==========================================
# predict_tool 代码
# ==========================================

# 全局推理计数器
INFER_COUNT = 0

def get_infer_count():
    global INFER_COUNT
    return INFER_COUNT

def reset_infer_count():
    global INFER_COUNT
    INFER_COUNT = 0

def model_predict(model, images):
    """
    模型预测函数，每调用一次自动累加当前批次图片数量作为推理次数
    """
    global INFER_COUNT
    batch_size = images.shape[0]
    INFER_COUNT += batch_size

    model.eval()
    with torch.no_grad():
        outputs = model(images)
        pred_labels = torch.argmax(outputs, dim=1)
    return pred_labels

# 配套模型加载包装类
class ModelWrapper:
    """
    AI模型统一接口占位类
    """
    def __init__(self):
        self.model = None
        self.predict_counter = 0

# ==========================================
# 3. eval_tool 所有函数
# ==========================================

# ==========================================
# Transfer Score 类 (ICLR 2024)
# 论文: "Can We Evaluate Domain Adaptation Models Without Target-Domain Labels?"
# 必须放在文件顶部，在其他函数之前
# ==========================================

class TransferScore:
    """
    Transfer Score - 无监督迁移评估指标

    三个组成部分:
    1. Uniformity: 分类器均匀性
    2. Hopkins Statistic: 特征聚类趋势
    3. Mutual Information: 预测可信度

    TS = -Uniformity + Hopkins + |MI| / ln(K)
    """

    def __init__(self, num_classes=10):
        self.num_classes = num_classes
        self.ideal_angle = self._compute_ideal_angle()

    def _compute_ideal_angle(self):
        K = self.num_classes
        return np.arccos(-1.0 / (K - 1))

    def compute_uniformity(self, classifier_weights):
        if not isinstance(classifier_weights, torch.Tensor):
            classifier_weights = torch.tensor(classifier_weights, dtype=torch.float32)

        weights = classifier_weights / torch.norm(classifier_weights, dim=0, keepdim=True)
        K = self.num_classes
        cos_sim = weights.T @ weights
        cos_sim = torch.clamp(cos_sim, -1.0, 1.0)
        angle_matrix = torch.arccos(cos_sim)

        ideal_matrix = torch.full((K, K), self.ideal_angle)
        ideal_matrix.fill_diagonal_(0)

        diff = angle_matrix - ideal_matrix
        mask = torch.triu(torch.ones(K, K), diagonal=1).bool()
        mse = (diff[mask] ** 2).mean()
        return mse.item()

    def compute_hopkins_statistic(self, features, n_samples=20):
        """
        计算 Hopkins 统计量（数值稳定版）
        """
        try:
            from sklearn.neighbors import NearestNeighbors
        except ImportError:
            return 0.5

        N, d = features.shape
        m = min(n_samples, N // 2)
        if m < 2:
            return 0.5

        # 采样
        indices = np.random.choice(N, m, replace=False)
        W = features[indices]
        min_vals = features.min(axis=0)
        max_vals = features.max(axis=0)
        U = np.random.uniform(min_vals, max_vals, (m, d))

        nbrs = NearestNeighbors(n_neighbors=2).fit(features)
        w_dist, _ = nbrs.kneighbors(W)
        w_dist = w_dist[:, 1]
        u_dist, _ = nbrs.kneighbors(U)
        u_dist = u_dist[:, 0]

        # ✅ 修复：用距离的平方和代替 d 次方（避免溢出）
        sum_u = (u_dist ** 2).sum()
        sum_w = (w_dist ** 2).sum()

        if sum_u + sum_w == 0:
            return 0.5
        return sum_u / (sum_u + sum_w)

    def compute_mutual_information(self, logits):
        try:
            from scipy.special import softmax
        except ImportError:
            probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
            entropies = -np.sum(probs * np.log(probs + 1e-10), axis=1)
            return max(0, np.log(self.num_classes) - entropies.mean())

        probs = softmax(logits, axis=1)
        entropies = -np.sum(probs * np.log(probs + 1e-10), axis=1)
        avg_entropy = entropies.mean()
        mean_probs = probs.mean(axis=0)
        entropy_mean = -np.sum(mean_probs * np.log(mean_probs + 1e-10))
        return entropy_mean - avg_entropy

    def compute_transfer_score(self, classifier_weights, target_features, target_logits):
        K = self.num_classes
        uniformity = self.compute_uniformity(classifier_weights)
        hopkins = self.compute_hopkins_statistic(target_features)
        mutual_info = self.compute_mutual_information(target_logits)
        transfer_score = -uniformity + hopkins + abs(mutual_info) / np.log(K)

        return {
            'transfer_score': transfer_score,
            'uniformity': uniformity,
            'hopkins_statistic': hopkins,
            'mutual_information': mutual_info,
            'interpretation': {
                'uniformity': '越小越好（分类器越均匀）',
                'hopkins': '越大越好（特征聚类越明显）',
                'mutual_info': '越大越好（预测越可信）',
                'transfer_score': '越大越好（整体迁移效果越好）'
            },
            'reference': 'Yang et al., ICLR 2024'
        }

def compute_migration_retention(adv_acc_raw, adv_acc_target):
    """
    计算迁移保持率
    Args:
        adv_acc_raw: 源模型对抗准确率
        adv_acc_target: 目标模型迁移后对抗准确率
    Returns:
        rate: 迁移保持率
        fail: 是否失效 (True/False)
    """
    if adv_acc_raw == 0:
        return 0.0, True
    rate = adv_acc_target / adv_acc_raw
    fail = rate < 0.9
    return rate, fail


def calc_fluctuation(acc_list):
    """计算波动幅度（标准差）"""
    return float(np.std(acc_list))


def evaluate_uda_with_transfer_score(
    source_model,
    target_model,
    target_loader,
    num_classes=10,
    device='cpu'
):
    """使用 Transfer Score 评估 UDA 模型"""

    target_model.eval()

    # ==========================================
    # 1. 正确提取分类器权重
    # ==========================================
    classifier_weights = None

    # 方法1：直接取 fc 层
    if hasattr(target_model, 'fc'):
        weights = target_model.fc.weight.data  # [10, 2048]
        classifier_weights = weights.cpu().numpy().T  # [2048, 10]
        print(f"  ✅ 从 fc 层提取权重: {classifier_weights.shape}")

    # 方法2：按名字找
    if classifier_weights is None:
        for name, param in target_model.named_parameters():
            if name == 'fc.weight':
                classifier_weights = param.data.cpu().numpy().T
                print(f"  ✅ 从 {name} 提取权重: {classifier_weights.shape}")
                break

    # 方法3：如果还是找不到，用最后一个线性层
    if classifier_weights is None:
        last_layer = list(target_model.children())[-1]
        if hasattr(last_layer, 'weight'):
            weights = last_layer.weight.data
            # 确保形状是 [特征维度, 类别数]
            if weights.shape[0] == num_classes:
                classifier_weights = weights.cpu().numpy().T
            else:
                classifier_weights = weights.cpu().numpy()
            print(f"  ✅ 从最后一层提取权重: {classifier_weights.shape}")

    if classifier_weights is None:
        return {'error': '无法提取分类器权重'}

    # ==========================================
    # 2. 强制修正维度
    # ==========================================
    # 确保 shape 是 [特征维度, 类别数]
    if classifier_weights.shape[1] != num_classes:
        # 如果转置后正确，就转置
        if classifier_weights.shape[0] == num_classes:
            classifier_weights = classifier_weights.T
            print(f"  ✅ 转置后: {classifier_weights.shape}")
        else:
            # 如果都不匹配，打印错误并返回
            print(f"  ❌ 权重维度错误: {classifier_weights.shape}，期望类别数 {num_classes}")
            return {'error': f'权重维度错误: {classifier_weights.shape}'}

    # ==========================================
    # 3. 提取特征（使用 hook）
    # ==========================================
    all_features = []
    all_logits = []

    # 注册 hook 提取 avgpool 层特征
    features = None
    def hook_fn(module, input, output):
        nonlocal features
        features = output.clone().detach()

    hook_handle = None
    for name, module in target_model.named_modules():
        if 'avgpool' in name:
            hook_handle = module.register_forward_hook(hook_fn)
            print(f"  ✅ 在 {name} 注册 hook")
            break

    if hook_handle is None:
        return {'error': '找不到 avgpool 层'}

    with torch.no_grad():
        for images, _ in target_loader:
            images = images.to(device)
            logits = target_model(images)
            all_logits.append(logits.cpu().numpy())

            if features is not None:
                # avgpool 输出 [batch, 2048, 1, 1] → flatten 成 [batch, 2048]
                feat = features.cpu().numpy().reshape(features.shape[0], -1)
                all_features.append(feat)
                features = None

    hook_handle.remove()

    if not all_features:
        return {'error': '无法提取特征'}

    all_logits = np.concatenate(all_logits, axis=0)
    all_features = np.concatenate(all_features, axis=0)

    print(f"  特征维度: {all_features.shape}")       # (1000, 2048)
    print(f"  分类器权重维度: {classifier_weights.shape}")  # (2048, 10)

    # ==========================================
    # 4. 验证维度匹配
    # ==========================================
    if all_features.shape[1] != classifier_weights.shape[0]:
        print(f"  ❌ 特征维度 {all_features.shape[1]} 与权重维度 {classifier_weights.shape[0]} 不匹配")
        return {'error': '维度不匹配'}

    # ==========================================
    # 5. 计算 Transfer Score
    # ==========================================
    ts = TransferScore(num_classes=num_classes)
    result = ts.compute_transfer_score(
        classifier_weights, all_features, all_logits
    )

    result['num_samples'] = len(all_features)
    result['num_classes'] = num_classes
    result['method'] = 'Transfer Score (ICLR 2024)'

    return result

# ==========================================
# 4. AttackGenerator
# ==========================================

class AttackGenerator:
    """
    对抗样本生成器（纯 PyTorch 实现）
    """

    def __init__(self, eps=0.05):
        self.eps = eps
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def _get_gradient(self, model, imgs, labels):
        """计算损失对输入的梯度"""
        imgs = imgs.to(self.device)
        labels = labels.to(self.device)

        imgs.requires_grad = True

        outputs = model(imgs)
        loss = nn.CrossEntropyLoss()(outputs, labels)

        model.zero_grad()
        loss.backward()

        return imgs.grad.data

    def generate_fgsm(self, imgs, labels, model):
        """FGSM 攻击（单步）"""
        imgs = imgs.to(self.device)
        labels = labels.to(self.device)

        grad = self._get_gradient(model, imgs, labels)
        perturbed = imgs + self.eps * grad.sign()
        return torch.clamp(perturbed, 0, 1)

    def generate_pgd(self, imgs, labels, model, steps=2, alpha=0.01):
        """PGD 攻击（多步）"""
        imgs = imgs.to(self.device)
        labels = labels.to(self.device)

        perturbed = imgs.clone().detach()

        for _ in range(steps):
            perturbed.requires_grad = True
            outputs = model(perturbed)
            loss = nn.CrossEntropyLoss()(outputs, labels)

            model.zero_grad()
            loss.backward()

            with torch.no_grad():
                grad = perturbed.grad.data
                perturbed = perturbed + alpha * grad.sign()
                perturbed = torch.where(
                    perturbed > imgs + self.eps,
                    imgs + self.eps,
                    perturbed
                )
                perturbed = torch.where(
                    perturbed < imgs - self.eps,
                    imgs - self.eps,
                    perturbed
                )
                perturbed = torch.clamp(perturbed, 0, 1)

        return perturbed

    def generate_bim(self, imgs, labels, model, steps=2):
        """BIM 攻击（Basic Iterative Method）"""
        imgs = imgs.to(self.device)
        labels = labels.to(self.device)

        alpha = self.eps / steps
        perturbed = imgs.clone().detach()

        for _ in range(steps):
            perturbed.requires_grad = True
            outputs = model(perturbed)
            loss = nn.CrossEntropyLoss()(outputs, labels)

            model.zero_grad()
            loss.backward()

            with torch.no_grad():
                grad = perturbed.grad.data
                perturbed = perturbed + alpha * grad.sign()
                perturbed = torch.clamp(perturbed, 0, 1)

        return perturbed

    def generate_all(self, imgs, labels, model):
        """生成所有攻击方法的对抗样本"""
        imgs = imgs.to(self.device)
        labels = labels.to(self.device)

        return {
            'fgsm': self.generate_fgsm(imgs, labels, model),
            'pgd': self.generate_pgd(imgs, labels, model),
            'bim': self.generate_bim(imgs, labels, model),
        }


# ==========================================
# 装饰器：性能监控
# ==========================================
def timing_decorator(func):
    """装饰器：记录函数执行时间"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        wrapper.last_execution_time = elapsed
        if args and hasattr(args[0], '_timings'):
            if not hasattr(args[0], '_timings'):
                args[0]._timings = {}
            args[0]._timings[func.__name__] = elapsed
        return result
    return wrapper

@contextmanager
def timing_context(func_name="unknown"):
    """上下文管理器：用于代码块计时"""
    start = time.time()
    try:
        yield
    finally:
        elapsed = time.time() - start
        print(f"⏱️ {func_name} 耗时: {elapsed:.3f}秒")
        if not hasattr(timing_context, 'timings'):
            timing_context.timings = {}
        timing_context.timings[func_name] = elapsed

def get_all_timings():
    """获取所有计时记录"""
    timings = {}
    if hasattr(timing_context, 'timings'):
        timings.update(timing_context.timings)
    return timings

def print_timing_summary():
    """打印计时汇总"""
    timings = get_all_timings()
    if not timings:
        print("⏱️ 暂无计时记录")
        return
    print("\n" + "=" * 60)
    print("⏱️ 性能计时汇总")
    print("=" * 60)
    total_time = 0
    for func_name, elapsed in timings.items():
        print(f"  {func_name}: {elapsed:.3f}秒")
        total_time += elapsed
    print("-" * 60)
    print(f"  总耗时: {total_time:.3f}秒")
    if total_time > 300:
        print(f"  ⚠️ 总耗时 {total_time:.1f}秒 超过5分钟限制！")
    else:
        print(f"  ✅ 总耗时 {total_time:.1f}秒，符合≤300秒要求")
    print("=" * 60)

# ==========================================
# 配置部分
# ==========================================
METRICS_CONFIG = {
    'thresholds': {
        'performance_retention': 0.9,
        'migration_retention': 0.9,
        'migration_stability': 0.05,
        'worst_case_accuracy': 0.7,
        'clean_accuracy': 0.85,
        'attack_accuracy': 0.7,
        'performance_degradation': 0.3,
        'worst_migration_acc': 0.85
    },
    'requirements': {
        'migration': ['migration_retention', 'migration_stability', 'worst_migration_acc'],
        'robustness': ['performance_retention', 'performance_degradation', 'worst_case_accuracy']
    }
}

def load_metrics_config(config_path=None):
    """加载指标配置文件"""
    if config_path:
        with open(config_path, 'r') as f:
            return json.load(f)
    return METRICS_CONFIG

# ==========================================
# 核心指标计算函数
# ==========================================

# ✅ 1. compute_accuracy - 只保留这一个版本
def compute_accuracy(origin_pred, adv_pred):
    """
    计算攻击成功率
    返回: (攻击成功率, 翻转样本数, 总样本数, 干净样本准确率, 对抗样本准确率)
    """
    origin_pred = np.array(origin_pred)
    adv_pred = np.array(adv_pred)

    total = len(origin_pred)

    # 1. 翻转样本数
    diff = origin_pred != adv_pred
    flip_count = diff.sum()

    # 2. 攻击成功率
    attack_rate = flip_count / total if total > 0 else 0.0

    # 3. 干净样本准确率（假设干净样本预测都是正确的）
    # 这里需要传入真实标签才能计算准确率，但你的函数没有 labels
    # 临时处理：假设 origin_pred 就是正确标签
    clean_acc = 1.0  # 默认100%

    # 4. 对抗样本准确率
    adv_acc_raw = (adv_pred == origin_pred).sum() / total if total > 0 else 0.0

    return attack_rate, int(flip_count), total, clean_acc, adv_acc_raw

# ✅ 2. sample_data - 只保留这一个版本
def sample_data(images, labels, ratio=0.1):
    """数据采样：从数据集中抽取 ratio 比例的数据"""
    num_samples = len(images)
    sample_count = int(num_samples * ratio)
    indices = np.random.choice(num_samples, sample_count, replace=False)
    sampled_images = images[indices]
    sampled_labels = labels[indices]
    return sampled_images, sampled_labels

# ✅ 3. compute_all_metrics - 核心函数，不要被覆盖！
def compute_all_metrics(clean_acc, attack_acc, all_attack_accs=None):
    """
    计算所有评价指标（攻击鲁棒性）
    这是被 batch_compute_attack_metrics 调用的核心函数
    """
    retention = attack_acc / clean_acc if clean_acc > 0 else 0.0
    degradation = 1.0 - retention
    worst_case = float(min(all_attack_accs)) if all_attack_accs else float(attack_acc)

    return {
        "clean_accuracy": float(clean_acc),
        "attack_accuracy": float(attack_acc),
        "performance_retention": float(retention),
        "performance_degradation": float(degradation),
        "worst_case_accuracy": float(worst_case),
    }

# ✅ 4. compute_robustness_metrics - 增强版（使用 compute_all_metrics）
def compute_robustness_metrics(clean_acc, attack_accs_dict, eps_values=None):
    """
    计算完整的鲁棒性指标（使用 compute_all_metrics）
    包含 AUC、安全裕度等新增指标
    """
    all_accs = []
    for attack_type, accs in attack_accs_dict.items():
        if isinstance(accs, dict):
            all_accs.extend(list(accs.values()))
        else:
            all_accs.append(accs)

    worst_case = min(all_accs) if all_accs else 0
    avg_case = np.mean(all_accs) if all_accs else 0
    degradation = 1 - (avg_case / clean_acc) if clean_acc > 0 else 0

    # ⚠️ 修复：鲁棒性AUC计算
    auc = 0.0
    if eps_values and len(eps_values) > 1:
        sorted_eps = sorted(eps_values)
        sorted_accs = []

        # 收集对应的准确率
        for eps in sorted_eps:
            found = False
            # 尝试从攻击结果中获取对应eps的准确率
            for attack_type, accs in attack_accs_dict.items():
                if isinstance(accs, dict):
                    # 检查多种可能的键名格式
                    key_candidates = [f'eps_{eps}', f'{eps}', f'eps{eps}']
                    for key in key_candidates:
                        if key in accs:
                            sorted_accs.append(accs[key])
                            found = True
                            break
                    if found:
                        break
                elif isinstance(accs, (int, float)):
                    # 如果只有一个值，所有eps都用同一个值
                    sorted_accs.append(accs)
                    found = True
                    break
            if not found:
                # 如果没找到，使用平均值
                sorted_accs.append(avg_case)

        # 确保长度一致
        if len(sorted_accs) == len(sorted_eps) and len(sorted_accs) > 1:
            # ⚠️ 关键修复：确保两个数组长度相同
            sorted_accs = np.array(sorted_accs[:len(sorted_eps)])
            sorted_eps = np.array(sorted_eps[:len(sorted_accs)])

            # 使用梯形法则计算AUC
            try:
                auc = np.trapz(sorted_accs, sorted_eps) / (max(sorted_eps) - min(sorted_eps))
            except ValueError:
                # 如果还是出错，使用简单平均
                auc = np.mean(sorted_accs) / clean_acc if clean_acc > 0 else 0
        else:
            auc = avg_case / clean_acc if clean_acc > 0 else 0
    else:
        auc = avg_case / clean_acc if clean_acc > 0 else 0

    safety_margin = worst_case / clean_acc if clean_acc > 0 else 0

    return {
        "clean_accuracy": clean_acc,
        "attack_accuracy": avg_case,
        "performance_retention": avg_case / clean_acc if clean_acc > 0 else 0,
        "performance_degradation": degradation,
        "worst_case_accuracy": worst_case,
        "robustness_auc": auc,
        "safety_margin": safety_margin,
        "attack_robustness_score": (avg_case / clean_acc) * (1 - degradation) if clean_acc > 0 else 0,
        "worst_case_retention": worst_case / clean_acc if clean_acc > 0 else 0,
    }

# ✅ 5. compute_migration_metrics - 基础版（只保留一个）
def compute_migration_metrics(source_acc, target_acc, history_accs=None):
    """计算迁移学习评估指标（基础版）"""
    retention = target_acc / source_acc if source_acc > 0 else 0.0
    stability = float(np.std(history_accs)) if history_accs and len(history_accs) > 1 else 0.0
    worst = float(min(history_accs)) if history_accs else float(target_acc)
    failed = retention < 0.9

    return {
        "migration_retention": float(retention),
        "migration_stability": float(stability),
        "worst_migration_acc": float(worst),
        "migration_failed": bool(failed),
    }

# ✅ 6. compute_comprehensive_migration_metrics - 增强版（新函数名，不覆盖上面的）
def compute_comprehensive_migration_metrics(source_acc, target_acc, history_accs=None):
    """
    增强版迁移学习指标（包含更多分析）
    注意：这个函数名与 compute_migration_metrics 不同，不会覆盖
    """
    retention = target_acc / source_acc if source_acc > 0 else 0
    stability = np.std(history_accs) if history_accs and len(history_accs) > 1 else 0.0
    stability_percent = stability * 100
    stability_score = 1.0 - min(stability / 0.05, 1.0) if stability > 0 else 1.0
    worst = min(history_accs) if history_accs else target_acc
    failed = (retention < 0.9) or (stability_percent > 5.0)
    performance_change = target_acc - source_acc
    adaptation_score = (retention + stability_score) / 2

    if history_accs and len(history_accs) > 1:
        trend = np.polyfit(range(len(history_accs)), history_accs, 1)[0]
        trend_direction = "improving" if trend > 0 else "degrading" if trend < 0 else "stable"
    else:
        trend = None
        trend_direction = "unknown"

    recovery_capability = 1.0 if retention > 0.95 else (retention / 0.95 if retention > 0 else 0)

    failure_reasons = []
    if retention < 0.9:
        failure_reasons.append(f"保持率 {retention:.2%} < 90%")
    if stability_percent > 5.0:
        failure_reasons.append(f"稳定性 ±{stability_percent:.2f}% > ±5%")

    return {
        "migration_retention": retention,
        "migration_stability": stability,
        "migration_stability_percent": stability_percent,
        "worst_migration_acc": worst,
        "migration_failed": failed,
        "failure_reasons": failure_reasons,
        "is_retention_compliant": retention >= 0.9,
        "is_stability_compliant": stability_percent <= 5.0,
        "is_migration_successful": not failed,
        "performance_change": performance_change,
        "performance_change_percent": (performance_change / source_acc * 100) if source_acc > 0 else 0,
        "adaptation_score": adaptation_score,
        "migration_trend": trend,
        "trend_direction": trend_direction,
        "recovery_capability": recovery_capability,
    }

# ✅ 7. batch_compute_attack_metrics - 只保留一个版本
def batch_compute_attack_metrics(clean_acc, attack_results):
    """批量计算多个攻击强度下的鲁棒性指标"""
    results = {}
    all_accs = []

    for result in attack_results:
        attack_name = result['attack_name']
        attack_acc = result['attack_acc']
        all_accs.append(float(attack_acc))

        # ✅ 调用 compute_all_metrics（确保这个函数存在且正确）
        metrics = compute_all_metrics(clean_acc, attack_acc, all_accs)
        results[attack_name] = {
            'attack_accuracy': float(attack_acc),
            'performance_retention': metrics['performance_retention'],
            'performance_degradation': metrics['performance_degradation'],
            'params': result.get('params', {})
        }

    results['summary'] = {
        'worst_case_accuracy': float(min(all_accs)),
        'avg_attack_accuracy': float(np.mean(all_accs)),
        'num_attacks': len(attack_results)
    }

    return results

# ✅ 8. compute_comprehensive_robustness_metrics - 只保留一个版本
def compute_comprehensive_robustness_metrics(clean_acc, attack_accs_by_type):
    """计算全面的鲁棒性指标"""
    results = {}
    all_accs = []
    attack_type_summaries = {}

    for attack_type, accs in attack_accs_by_type.items():
        attack_acc_list = list(accs.values())
        all_accs.extend(attack_acc_list)

        attack_type_summaries[attack_type] = {
            'min_acc': min(attack_acc_list),
            'max_acc': max(attack_acc_list),
            'avg_acc': np.mean(attack_acc_list),
            'std_acc': np.std(attack_acc_list),
            'num_eps': len(attack_acc_list)
        }

        for eps, acc in accs.items():
            # ✅ 调用 compute_all_metrics
            metrics = compute_all_metrics(clean_acc, acc, attack_acc_list)
            metrics['attack_type'] = attack_type
            metrics['eps'] = eps
            results[f"{attack_type}_{eps}"] = metrics

    if all_accs:
        worst_case = min(all_accs)
        avg_case = np.mean(all_accs)
        max_degradation = max([1 - acc/clean_acc for acc in all_accs]) if clean_acc > 0 else 0
        robustness_score = (avg_case / clean_acc) * (1 - max_degradation) if clean_acc > 0 else 0
        safety_margin = worst_case / clean_acc if clean_acc > 0 else 0

        results['global_summary'] = {
            'global_worst_case': worst_case,
            'global_avg_case': avg_case,
            'max_degradation': max_degradation,
            'num_attacks_tested': len(all_accs),
            'robustness_score': robustness_score,
            'safety_margin': safety_margin,
            'attack_type_summaries': attack_type_summaries
        }

    return results

# ==========================================
# PerformanceMonitor 类
# ==========================================
class PerformanceMonitor:
    """计算效率监控（确保所有约束达标）"""
    def __init__(self, max_inference=1000, max_time=300):
        self.max_inference = max_inference
        self.max_time = max_time
        self.inference_count = 0
        self.start_time = None
        self.end_time = None
        self.run_history = []
        self.prediction_times = []

    def start(self):
        self.start_time = datetime.now()
        self.inference_count = 0
        self.prediction_times = []

    def count_inference(self, n=1, time_cost=None):
        self.inference_count += n
        if time_cost:
            self.prediction_times.append(time_cost)
        if self.inference_count > self.max_inference:
            print(f"⚠️ 推理次数 {self.inference_count} 超过限制 {self.max_inference}")
            return False
        return True

    def stop(self):
        self.end_time = datetime.now()
        elapsed = (self.end_time - self.start_time).total_seconds()
        self.run_history.append({
            'inference_count': self.inference_count,
            'elapsed_time': elapsed,
            'timestamp': self.end_time.isoformat(),
            'avg_prediction_time': np.mean(self.prediction_times) if self.prediction_times else 0
        })
        consistency = self.get_consistency_score()
        return {
            "inference_count": self.inference_count,
            "elapsed_time": elapsed,
            "consistency_score": consistency,
            "is_inference_compliant": self.inference_count <= self.max_inference,
            "is_time_compliant": elapsed <= self.max_time,
            "is_consistent": consistency >= 0.95,
            "all_compliant": (
                self.inference_count <= self.max_inference and
                elapsed <= self.max_time and
                consistency >= 0.95
            )
        }

    def get_consistency_score(self):
        if len(self.run_history) < 2:
            return 1.0
        times = [run['elapsed_time'] for run in self.run_history]
        counts = [run['inference_count'] for run in self.run_history]
        time_cv = np.std(times) / np.mean(times) if np.mean(times) > 0 else 0
        count_cv = np.std(counts) / np.mean(counts) if np.mean(counts) > 0 else 0
        consistency = 1 - (time_cv + count_cv) / 2
        return max(0, min(1, consistency))

    def get_report(self):
        if not self.run_history:
            return "暂无运行记录"
        latest = self.run_history[-1]
        report = f"""
╔══════════════════════════════════════════════════════════════╗
║                    性能监控报告                               ║
╠══════════════════════════════════════════════════════════════╣
║  推理次数:        {latest['inference_count']:>6} 次  {'✅' if latest['inference_count'] <= 1000 else '❌'}   ║
║  执行时间:        {latest['elapsed_time']:>6.2f} 秒  {'✅' if latest['elapsed_time'] <= 300 else '❌'}   ║
║  一致性得分:      {self.get_consistency_score():>6.2%}  {'✅' if self.get_consistency_score() >= 0.95 else '❌'}   ║
╠══════════════════════════════════════════════════════════════╣
║  运行次数:        {len(self.run_history):>6} 次              ║
║  总推理次数:      {sum(r['inference_count'] for r in self.run_history):>6} 次   ║
║  平均时间:        {np.mean([r['elapsed_time'] for r in self.run_history]):>6.2f} 秒   ║
╚══════════════════════════════════════════════════════════════╝
"""
        return report

# ==========================================
# 阈值检查和表格格式化
# ==========================================
def check_threshold(category, metric_name, value):
    """检查指标是否满足阈值要求"""
    thresholds = {
        'performance_retention': 0.9,
        'migration_retention': 0.9,
        'migration_stability': 0.05,
        'worst_case_accuracy': 0.7,
        'clean_accuracy': 0.85,
        'attack_accuracy': 0.7,
        'performance_degradation': 0.3,
    }

    if metric_name in thresholds:
        if metric_name in ['migration_stability', 'performance_degradation']:
            return value <= thresholds[metric_name]
        else:
            return value >= thresholds[metric_name]
    return None

def format_metrics_to_table(metrics_dict, attack_type=None):
    """将指标格式化为标准化表格"""
    rows = []
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for key, value in metrics_dict.items():
        if isinstance(value, dict):
            for sub_key, sub_value in value.items():
                if isinstance(sub_value, dict):
                    for sub_sub_key, sub_sub_value in sub_value.items():
                        rows.append({
                            'timestamp': timestamp,
                            'attack_type': attack_type or 'N/A',
                            'metric_category': f"{key}.{sub_key}",
                            'metric_name': sub_sub_key,
                            'metric_value': sub_sub_value,
                            'is_threshold_met': check_threshold(f"{key}.{sub_key}", sub_sub_key, sub_sub_value)
                        })
                else:
                    rows.append({
                        'timestamp': timestamp,
                        'attack_type': attack_type or 'N/A',
                        'metric_category': key,
                        'metric_name': sub_key,
                        'metric_value': sub_value,
                        'is_threshold_met': check_threshold(key, sub_key, sub_value)
                    })
        else:
            rows.append({
                'timestamp': timestamp,
                'attack_type': attack_type or 'N/A',
                'metric_category': 'base',
                'metric_name': key,
                'metric_value': value,
                'is_threshold_met': check_threshold('base', key, value)
            })

    return pd.DataFrame(rows)

# ==========================================
# Excel导出功能
# ==========================================
def flatten_metrics_dict(d, parent_key='', sep='_'):
    """扁平化嵌套字典"""
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_metrics_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def calculate_summary_statistics(metrics_list):
    """计算汇总统计"""
    if not metrics_list:
        return {}
    df = pd.DataFrame(metrics_list)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    summary = {}
    for col in numeric_cols:
        summary[f'{col}_mean'] = df[col].mean()
        summary[f'{col}_std'] = df[col].std()
        summary[f'{col}_min'] = df[col].min()
        summary[f'{col}_max'] = df[col].max()
    return summary

def check_all_thresholds(metrics_list):
    """检查所有指标是否满足阈值"""
    results = []
    thresholds = METRICS_CONFIG['thresholds']

    for i, metrics in enumerate(metrics_list):
        for key, value in metrics.items():
            if key in thresholds:
                threshold = thresholds[key]
                if key in ['migration_stability', 'performance_degradation']:
                    is_met = value <= threshold
                else:
                    is_met = value >= threshold
                results.append({
                    'test_id': i,
                    'metric': key,
                    'value': value,
                    'threshold': threshold,
                    'met': is_met,
                    'status': '✅ 达标' if is_met else '❌ 不达标'
                })
    return results

def generate_compliance_summary(threshold_check):
    """生成合规性总结"""
    if not threshold_check:
        return {'total_metrics': 0, 'passed': 0, 'failed': 0, 'all_passed': False}

    total = len(threshold_check)
    passed = sum(1 for item in threshold_check if item['met'])
    failed = total - passed

    return {
        'total_metrics': total,
        'passed': passed,
        'failed': failed,
        'pass_rate': passed / total if total > 0 else 0,
        'all_passed': failed == 0,
        'check_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

def export_metrics_to_excel(metrics_data, output_path=None, output_dir="results"):
    """导出指标台账到Excel"""
    os.makedirs(output_dir, exist_ok=True)

    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(output_dir, f"metrics_ledger_{timestamp}.xlsx")

    if isinstance(metrics_data, dict):
        metrics_list = [flatten_metrics_dict(metrics_data)]
    elif isinstance(metrics_data, list):
        metrics_list = metrics_data
    else:
        raise ValueError("metrics_data 必须是 dict 或 list")

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_main = pd.DataFrame(metrics_list)
        df_main.to_excel(writer, sheet_name='Main_Metrics', index=False)

        summary = calculate_summary_statistics(metrics_list)
        if summary:
            pd.DataFrame([summary]).to_excel(writer, sheet_name='Summary', index=False)

        threshold_check = check_all_thresholds(metrics_list)
        if threshold_check:
            df_threshold = pd.DataFrame(threshold_check)
            df_threshold.to_excel(writer, sheet_name='Threshold_Check', index=False)

        compliance_summary = generate_compliance_summary(threshold_check)
        pd.DataFrame([compliance_summary]).to_excel(writer, sheet_name='Compliance', index=False)

    print(f"✅ Excel台账已导出: {output_path}")
    return output_path

# ==========================================
# 完整台账生成
# ==========================================
def generate_complete_ledger(
    clean_acc,
    attack_results,
    source_acc=None,
    target_acc=None,
    inference_count=None,
    data_ratio=None,
    elapsed_time=None,
    output_dir="results"
):
    """生成完整的测试台账"""
    os.makedirs(output_dir, exist_ok=True)

    attack_metrics = batch_compute_attack_metrics(clean_acc, attack_results)

    migration_metrics = None
    if source_acc is not None and target_acc is not None:
        migration_metrics = compute_migration_metrics(source_acc, target_acc)

    ledger = {
        'test_info': {
            'timestamp': datetime.now().isoformat(),
            'inference_count': inference_count,
            'data_ratio': data_ratio,
            'elapsed_time': elapsed_time
        },
        'migration_metrics': migration_metrics,
        'attack_metrics': attack_metrics,
        'compliance': {
            'inference_compliant': inference_count <= 1000 if inference_count else None,
            'data_compliant': data_ratio <= 0.10 if data_ratio else None,
            'time_compliant': elapsed_time <= 300 if elapsed_time else None
        }
    }

    excel_path = export_metrics_to_excel(ledger, output_dir=output_dir)

    json_path = os.path.join(output_dir, f"ledger_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    with open(json_path, 'w') as f:
        json.dump(ledger, f, indent=2)

    print(f"✅ 完整台账已生成:")
    print(f"   Excel: {excel_path}")
    print(f"   JSON: {json_path}")

    return ledger

# ==========================================
# 线程安全的指标收集器
# ==========================================
class MetricsCollector:
    """线程安全的指标收集器"""
    def __init__(self):
        self._lock = threading.Lock()
        self.metrics = defaultdict(list)
        self._start_time = datetime.now()

    def add_metrics(self, test_id, metrics_dict):
        with self._lock:
            self.metrics[test_id].append({
                'timestamp': datetime.now().isoformat(),
                'test_id': str(test_id),
                **metrics_dict
            })

    def add_attack_result(self, test_id, attack_name, attack_acc, clean_acc, params=None):
        metrics = compute_all_metrics(clean_acc, attack_acc, [attack_acc])
        metrics.update({
            'attack_name': attack_name,
            'attack_accuracy': attack_acc,
            'clean_accuracy': clean_acc,
            'params': str(params) if params else '{}'
        })
        self.add_metrics(test_id, metrics)

    def get_all_metrics(self):
        with self._lock:
            return {k: v.copy() for k, v in self.metrics.items()}

    def get_flat_metrics(self):
        all_metrics = self.get_all_metrics()
        flat_list = []
        for test_id, metrics_list in all_metrics.items():
            for metrics in metrics_list:
                flat_list.append(metrics)
        return flat_list

    def get_summary(self):
        flat_list = self.get_flat_metrics()
        if not flat_list:
            return {'total_tests': 0, 'total_metrics': 0, 'status': 'no_data'}

        df = pd.DataFrame(flat_list)
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        summary = {
            'total_tests': len(df['test_id'].unique()),
            'total_metrics': len(df),
            'start_time': self._start_time.isoformat(),
            'end_time': datetime.now().isoformat()
        }

        for col in numeric_cols:
            if col not in ['test_id']:
                summary[f'{col}_mean'] = float(df[col].mean())
                summary[f'{col}_std'] = float(df[col].std())
                summary[f'{col}_min'] = float(df[col].min())
                summary[f'{col}_max'] = float(df[col].max())

        return summary

    def export_to_excel(self, output_path=None, output_dir="results"):
        """导出所有收集的指标到Excel"""
        os.makedirs(output_dir, exist_ok=True)

        if output_path is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_path = os.path.join(output_dir, f"collector_metrics_{timestamp}.xlsx")

        flat_list = self.get_flat_metrics()
        if not flat_list:
            print("⚠️ 没有收集到任何指标")
            return None

        df = pd.DataFrame(flat_list)

        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='All_Metrics', index=False)

            summary = self.get_summary()
            pd.DataFrame([summary]).to_excel(writer, sheet_name='Summary', index=False)

            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if 'test_id' in numeric_cols:
                numeric_cols.remove('test_id')

            if numeric_cols:
                grouped = df.groupby('test_id')[numeric_cols].agg(['mean', 'std', 'min', 'max'])
                grouped.to_excel(writer, sheet_name='Grouped_Stats')
            else:
                count_stats = df.groupby('test_id').size().reset_index(name='record_count')
                count_stats.to_excel(writer, sheet_name='Record_Counts', index=False)

            overview = pd.DataFrame({
                '总测试数': [len(df['test_id'].unique())],
                '总记录数': [len(df)],
                '数值列': [', '.join(numeric_cols) if numeric_cols else '无'],
                '导出时间': [datetime.now().strftime("%Y-%m-%d %H:%M:%S")]
            })
            overview.to_excel(writer, sheet_name='Overview', index=False)

        print(f"✅ 收集器数据已导出: {output_path}")
        return output_path

    def clear(self):
        with self._lock:
            self.metrics.clear()
            self._start_time = datetime.now()

    def get_test_count(self):
        with self._lock:
            return len(self.metrics)

    def get_metric_count(self):
        with self._lock:
            return sum(len(v) for v in self.metrics.values())

# ==========================================
# 完整测试流程（带计时）
# ==========================================
@timing_decorator
def run_full_test_pipeline(clean_acc, attack_results, source_acc=None, target_acc=None):
    """运行完整的测试流程（带计时）"""
    print("🚀 开始完整测试流程...")

    with timing_context("attack_metrics"):
        attack_metrics = batch_compute_attack_metrics(clean_acc, attack_results)

    migration_metrics = None
    if source_acc is not None and target_acc is not None:
        with timing_context("migration_metrics"):
            migration_metrics = compute_migration_metrics(source_acc, target_acc)

    result = {
        'attack_metrics': attack_metrics,
        'migration_metrics': migration_metrics,
        'timings': get_all_timings(),
        'total_time': sum(get_all_timings().values())
    }

    print_timing_summary()

# 全局配置
MAX_INFER_LIMIT = 1000
MAX_TIME_LIMIT = 300
TEST_ROUND = 1
SAFE_THRESHOLD = 0.70
EPS_LIST = [0.01, 0.03, 0.05, 0.08, 0.10]
ATTACK_LIST = ["fgsm", "pgd", "bim"]
PER_EPS_REPEAT = 1  # 重复测试次数

# 二阶差分求拐点
def find_inflection_point(x_arr, y_arr):
    x = np.array(x_arr)
    y = np.array(y_arr)
    dy1 = np.diff(y) / np.diff(x)
    dy2 = np.diff(dy1)
    idx = np.argmax(np.abs(dy2)) + 1
    return x[idx], y[idx], idx

def run_single_test(attack, eps, imgs, labels, raw_model, target_model, device):
    attacker = AttackGenerator(eps=eps)
    # 干净样本推理：仅推理关闭梯度，提速
    with torch.no_grad():
        clean_pred = model_predict(raw_model, imgs)
    curr_infer = get_infer_count()
    print(f"\n推理批次：本批16张，累计推理总数：{curr_infer}")

    print(f"扰动强度eps={eps}，执行{attack.upper()}对抗样本生成")
    # 对抗样本生成必须开启梯度，不能加no_grad
    adv_imgs = getattr(attacker, f"generate_{attack}")(imgs, labels, raw_model)

    # 对抗推理关闭梯度
    with torch.no_grad():
        adv_pred_raw = model_predict(raw_model, adv_imgs)
        adv_pred_target = model_predict(target_model, adv_imgs)

    attack_rate, flip_num, total, clean_acc, adv_acc_raw = compute_accuracy(clean_pred, adv_pred_raw)
    _, _, _, _, adv_acc_target = compute_accuracy(clean_pred, adv_pred_target)
    migrate_rate, migrate_fail = compute_migration_retention(adv_acc_raw, adv_acc_target)
    over_safe = bool(adv_acc_raw >= SAFE_THRESHOLD)

    print("-------本轮测试指标汇总-------")
    print(f"扰动强度：{eps} | 攻击算法：{attack.upper()}")
    print(f"1.正常输入基准准确率：{clean_acc:.4f}")
    print(f"2.攻击下有效性能【源模型对抗准确率】：{adv_acc_raw:.4f}")
    print(f"3.目标模型迁移后对抗准确率：{adv_acc_target:.4f}")
    print(f"4.性能退化幅度(攻击成功率)：{attack_rate:.4f}")
    print(f"5.迁移性能保持率：{migrate_rate:.4f}（合格线≥0.9）")
    print(f"6.迁移失效标记：{migrate_fail}")
    print(f"7.强扰动安全阈值0.7，当前是否达标：{over_safe}")
    if not over_safe:
        print(f"   提示：源模型对抗准确率{adv_acc_raw:.4f} < {SAFE_THRESHOLD}，模型鲁棒性不足")
    print(f"8.预测翻转样本：{flip_num}/{total}")
    print(f"批次结束总推理次数：{get_infer_count()}")
    print("------------------------------")

    return {
        "eps": eps,
        "attack": attack,
        "clean_acc": clean_acc,
        "source_adv_acc": adv_acc_raw,
        "target_adv_acc": adv_acc_target,
        "attack_success": attack_rate,
        "migrate_keep": migrate_rate,
        "migrate_fail": migrate_fail,
        "safe_pass": over_safe,
        "flip_samples": f"{flip_num}/{total}"
    }

def main():
    total_start = time.time()
    device = torch.device("cpu")
    print("运行设备：CPU")
    print(f"资源约束：推理调用≤{MAX_INFER_LIMIT}次，总运行时长≤{MAX_TIME_LIMIT}秒")
    reset_infer_count()

   # 加载预训练模型
    raw_model = resnet50(weights=ResNet50_Weights.DEFAULT)

    # ⚠️ 关键：把 fc 层从 1000 类改成 10 类
    raw_model.fc = torch.nn.Linear(2048, 10)
    try:
        raw_model.load_state_dict(torch.load("resnet50_cifar10.pth", map_location='cpu'))
        print("✅ 加载训练好的权重")
    except:
        print("⚠️ 未找到训练好的权重，使用随机初始化（建议先训练模型）")

    raw_model = raw_model.to(device).eval()

    # 目标模型用同样的方式
    target_model = resnet50(weights=ResNet50_Weights.DEFAULT)
    target_model.fc = torch.nn.Linear(2048, 10)
    try:
        target_model.load_state_dict(torch.load("resnet50_cifar10.pth", map_location='cpu'))
        print("✅ 目标模型加载训练好的权重")
    except:
        print("⚠️ 目标模型使用随机初始化")
    target_model = target_model.to(device).eval()
    test_loader = load_cifar10(batch_size=16)
    test_iter = iter(test_loader)

    global_summary = []
    eps_record = {eps: [] for eps in EPS_LIST}

    for round_idx in range(TEST_ROUND):
        run_time = time.time() - total_start
        if run_time >= MAX_TIME_LIMIT or get_infer_count() >= MAX_INFER_LIMIT:
            print("【触发约束】资源上限到达，提前终止实验，保存现有数据")
            try:
                with open("result_log.json", "w", encoding="utf-8") as f:
                    json.dump(global_summary, f, ensure_ascii=False, indent=2)
                print("临时数据保存成功：result_log.json")
            except Exception as e:
                print(f"保存JSON文件失败，错误信息：{e}")
            return
        print(f"\n===== 第{round_idx+1}轮完整测试 =====")
        for eps in EPS_LIST:
            for repeat in range(PER_EPS_REPEAT):
                print(f"\n---- 扰动强度eps={eps} 重复测试{repeat+1}/{PER_EPS_REPEAT} ----")
                try:
                    imgs, labels = next(test_iter)
                except StopIteration:
                    test_iter = iter(test_loader)
                    imgs, labels = next(test_iter)
                imgs, labels = imgs.to(device), labels.to(device)
                for attack_name in ATTACK_LIST:
                    res_data = run_single_test(attack_name, eps, imgs, labels, raw_model, target_model, device)
                    global_summary.append(res_data)
                    eps_record[eps].append(res_data["source_adv_acc"])
                    if time.time() - total_start >= MAX_TIME_LIMIT or get_infer_count() >= MAX_INFER_LIMIT:
                        print("【触发约束】资源超限，终止运行")
                        try:
                            with open("result_log.json", "w", encoding="utf-8") as f:
                                json.dump(global_summary, f, ensure_ascii=False, indent=2)
                            print("临时数据保存成功：result_log.json")
                        except Exception as e:
                            print(f"保存JSON文件失败，错误信息：{e}")
                        return

    # 后处理波动分析
    print("\n==========【后处理：扰动-准确率分析】==========")
    fluct_info = {}
    for eps, acc_list in eps_record.items():
        if len(acc_list) >= 2:
            fluct = calc_fluctuation(acc_list)
            fluct_info[eps] = {
                "acc_list": acc_list,
                "fluctuation": fluct,
                "is_ok": bool(fluct <= 0.05)  # 强制转为Python原生bool
            }
            print(f"eps={eps} | 波动幅度：{fluct:.4f} | 波动合规：{fluct <= 0.05}")

    # 拐点计算
    eps_x = EPS_LIST
    avg_acc_y = [np.mean(eps_record[e]) for e in EPS_LIST]
    inflect_x, inflect_y, _ = find_inflection_point(eps_x, avg_acc_y)
    print(f"\n扰动-准确率曲线拐点：eps={inflect_x:.4f}，对应对抗准确率={inflect_y:.4f}")

    # 绘图
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False
    plt.figure(figsize=(9, 5))
    plt.plot(eps_x, avg_acc_y, marker="o", linewidth=2, label="平均对抗准确率")
    plt.scatter(inflect_x, inflect_y, c="red", s=120, label=f"拐点 eps={inflect_x:.3f}")
    plt.axhline(y=SAFE_THRESHOLD, c="orange", linestyle="--", label=f"安全阈值 {SAFE_THRESHOLD}")
    plt.xlabel("扰动强度 eps")
    plt.ylabel("源模型平均对抗准确率")
    plt.title("扰动强度-模型鲁棒准确率关系曲线")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig("eps_acc_curve.png", dpi=300)
    plt.close()
    print("曲线图已保存 eps_acc_curve.png")

    # 输出完整报告
    final_report = {
        "time_cost_s": round(time.time() - total_start, 2),
        "total_infer_count": get_infer_count(),
        "max_infer_limit": MAX_INFER_LIMIT,
        "max_time_limit": MAX_TIME_LIMIT,
        "safe_threshold": SAFE_THRESHOLD,
        "eps_list": EPS_LIST,
        "attack_types": ATTACK_LIST,
        "fluctuation_analysis": fluct_info,
        "curve_inflection": {"eps": inflect_x, "acc": inflect_y},
        "all_test_data": global_summary
    }

    # ==========================================
    # Transfer Score 评估 (ICLR 2024)
    # ==========================================
    print("\n" + "=" * 60)
    print("📊 Transfer Score 评估 (无标签迁移评估)")
    print("=" * 60)

    try:


        # 使用已加载的 test_loader
        ts_result = evaluate_uda_with_transfer_score(
            source_model=raw_model,
            target_model=raw_model,  # 用源模型作为基准
            target_loader=test_loader,
            num_classes=10,
            device=device
        )

        print(f"  Transfer Score: {ts_result['transfer_score']:.4f}")
        print(f"    均匀性 (越小越好): {ts_result['uniformity']:.4f}")
        print(f"    Hopkins统计量 (越大越好): {ts_result['hopkins_statistic']:.4f}")
        print(f"    互信息 (越大越好): {ts_result['mutual_information']:.4f}")
        print(f"  参考: Yang et al., ICLR 2024")

        # 保存到 final_report
        final_report['transfer_score'] = ts_result

    except Exception as e:
        print(f"  ⚠️ Transfer Score 评估失败: {e}")

    # 在保存前修复
    def convert_to_serializable(obj):
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [convert_to_serializable(v) for v in obj]
        return obj

    # 保存时
    final_report_clean = convert_to_serializable(final_report)
    with open("final_report.json", "w", encoding="utf-8") as f:
        json.dump(final_report_clean, f, ensure_ascii=False, indent=2)

        print("\n==========全部实验执行完成==========")
        print(f"总运行耗时：{time.time()-total_start:.2f}s")
        print(f"总推理次数：{get_infer_count()}")

if __name__ == "__main__":
    main()

运行设备：CPU
资源约束：推理调用≤1000次，总运行时长≤300秒
✅ 加载训练好的权重
✅ 目标模型加载训练好的权重
完整数据集总量：10000，本次测试选用样本：1000，数据占比：10.00%

===== 第1轮完整测试 =====

---- 扰动强度eps=0.01 重复测试1/1 ----

推理批次：本批16张，累计推理总数：16
扰动强度eps=0.01，执行FGSM对抗样本生成
-------本轮测试指标汇总-------
扰动强度：0.01 | 攻击算法：FGSM
1.正常输入基准准确率：1.0000
2.攻击下有效性能【源模型对抗准确率】：0.1875
3.目标模型迁移后对抗准确率：0.1875
4.性能退化幅度(攻击成功率)：0.8125
5.迁移性能保持率：1.0000（合格线≥0.9）
6.迁移失效标记：False
7.强扰动安全阈值0.7，当前是否达标：False
   提示：源模型对抗准确率0.1875 < 0.7，模型鲁棒性不足
8.预测翻转样本：13/16
批次结束总推理次数：48
------------------------------

推理批次：本批16张，累计推理总数：64
扰动强度eps=0.01，执行PGD对抗样本生成
-------本轮测试指标汇总-------
扰动强度：0.01 | 攻击算法：PGD
1.正常输入基准准确率：1.0000
2.攻击下有效性能【源模型对抗准确率】：0.1250
3.目标模型迁移后对抗准确率：0.1250
4.性能退化幅度(攻击成功率)：0.8750
5.迁移性能保持率：1.0000（合格线≥0.9）
6.迁移失效标记：False
7.强扰动安全阈值0.7，当前是否达标：False
   提示：源模型对抗准确率0.1250 < 0.7，模型鲁棒性不足
8.预测翻转样本：14/16
批次结束总推理次数：96
------------------------------

推理批次：本批16张，累计推理总数：112
扰动强度eps=0.01，执行BIM对抗样本生成
-------本轮测试指标汇总-------
扰动强度：0.01 | 攻击算法：BIM
1.正常输入基准准确率：1.0000
2.攻击下有效性能【源模型对抗准确率】：0.1250
3.目标模型迁移后对抗准确率：0.12

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet50, ResNet50_Weights

# 1. 加载数据
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)

# 2. 加载模型，替换 fc 层
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"训练设备: {device}")

model = resnet50(weights=ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(2048, 10)
model = model.to(device)

# ✅ 全部层都可训练（不冻结）
for param in model.parameters():
    param.requires_grad = True

# 3. 优化器（用小学习率）
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

# 4. 训练
print("\n开始微调全部层...")
for epoch in range(5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = correct / total
    print(f"Epoch [{epoch+1}/5] Loss: {running_loss/len(train_loader):.4f}, Acc: {train_acc:.4f}")

# 5. 保存模型
torch.save(model.state_dict(), "resnet50_cifar10.pth")
print(f"\n✅ 训练完成！最终准确率: {train_acc:.4f}")
print("✅ 模型已保存: resnet50_cifar10.pth")

训练设备: cpu

开始微调全部层...


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet50, ResNet50_Weights

# 1. 加载数据
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

# 2. 加载模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"训练设备: {device}")

model = resnet50(weights=ResNet50_Weights.DEFAULT)

# ✅ 关键：冻结所有层，只训练 fc 层
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(2048, 10)
model = model.to(device)

# 3. 只优化 fc 层
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 4. 只训练 2 轮
print("\n开始训练 fc 层（只训练最后一层）...")
for epoch in range(2):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = correct / total
    print(f"Epoch [{epoch+1}/2] Loss: {running_loss/len(train_loader):.4f}, Acc: {train_acc:.4f}")

torch.save(model.state_dict(), "resnet50_cifar10.pth")
print(f"\n✅ 训练完成！最终准确率: {train_acc:.4f}")

训练设备: cpu

开始训练 fc 层（只训练最后一层）...


In [1]:
import torch
import os

file_path = "resnet50_cifar10.pth"
if os.path.exists(file_path):
    size = os.path.getsize(file_path) / 1024 / 1024
    print(f"✅ 文件存在，大小: {size:.2f} MB")
    try:
        state_dict = torch.load(file_path, map_location='cpu')
        print(f"✅ 模型加载成功！包含 {len(state_dict)} 个参数层")
    except Exception as e:
        print(f"❌ 加载失败: {e}")
else:
    print("❌ 文件不存在，请检查文件位置")

✅ 文件存在，大小: 90.06 MB
✅ 模型加载成功！包含 320 个参数层
